# 01 — Official Master-Plan Data Preparation

Prepare the adopted `master_v1` JSONL release for seeds **42, 123, 456**.
The implementation lives in `src/`; this notebook runs the shared modules.
Set `BASE_PATH_OVERRIDE` to the team's actual shared folder. The Colab default
is the plan's suggested path and must be checked against the folder you can access.

The original `01_data_preparation.ipynb` remains in place. Previous versions of
these workflow notebooks are available in Git history.

For Colab, set `BASE_PATH_OVERRIDE` in the setup cell to the actual mounted
`CS682_Neural_Project` folder containing `01_data`. The default is an example;
check its location in the file browser after Drive mounting. Keep the override
at the project root, without appending `/01_data`. See the README Colab setup.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

PROJECT_ROOT_OVERRIDE = None
INSTALL_DEPENDENCIES = IN_COLAB
BASE_PATH_OVERRIDE = "/content/drive/MyDrive/CS682_Neural_Project" if IN_COLAB else None

if PROJECT_ROOT_OVERRIDE:
    PROJECT_ROOT = Path(PROJECT_ROOT_OVERRIDE).expanduser().resolve()
else:
    PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                         if (p / "configs/experiment.yaml").is_file()), None)
    if PROJECT_ROOT is None and IN_COLAB:
        PROJECT_ROOT = Path("/content/CS682-Synthetic-Data-Project")
        if not PROJECT_ROOT.exists():
            subprocess.run(["git", "clone", "--branch", "experiment/b2-baselines",
                            "https://github.com/khushboopandey6/CS682-Synthetic-Data-Project.git",
                            str(PROJECT_ROOT)], check=True)
    if PROJECT_ROOT is None:
        raise RuntimeError("Set PROJECT_ROOT_OVERRIDE to the cloned project directory.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
if INSTALL_DEPENDENCIES:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    str(PROJECT_ROOT / "requirements.txt")], check=True)
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

from src.project_config import load_config, resolve_base_path
CONFIG_PATH = PROJECT_ROOT / "configs/experiment.yaml"
config = load_config(CONFIG_PATH)
BASE_PATH = resolve_base_path(config, BASE_PATH_OVERRIDE)
print("Protocol:", config["protocol_id"], "| Status:", config["status"])
print("Artifact base:", BASE_PATH)

## Prepare the pinned source and official subsets

A fresh Colab runtime can download the pinned public source. Local execution
reuses the verified cache. The test split is exported for later reporting;
it is never an input to the classifier pilot.

In [ ]:
from src.source_cache import ensure_source_cache
from src.data_preparation import prepare_official_data

DOWNLOAD_SOURCE = IN_COLAB
SOURCE_ROOT = PROJECT_ROOT / "data/reproducible"
ensure_source_cache(config, SOURCE_ROOT, allow_download=DOWNLOAD_SOURCE)
manifest = prepare_official_data(config, BASE_PATH, SOURCE_ROOT)
print("Seeds:", manifest["config"]["seeds"])
print("Rare IDs:", manifest["rare_class_ids"])
for name, metadata in manifest["datasets"].items():
    print(name, metadata["rows"])

## Prepare reproducible B2 and B3 datasets

B2 preserves all real examples and records repeated draws. Balanced B2 is an
exact B1 identity. B3 targets one EDA example per original and records shortfalls.
Negation and numeric-token safeguards are retained from the tested implementation.

In [ ]:
import nltk
from nltk.corpus import wordnet
from src.baselines import prepare_baseline_data

DOWNLOAD_WORDNET = IN_COLAB
CORPUS_ROOT = SOURCE_ROOT / "_cache/nltk_data"
nltk.data.path.insert(0, str(CORPUS_ROOT))
try:
    wordnet.ensure_loaded()
except LookupError:
    if not DOWNLOAD_WORDNET:
        raise RuntimeError("WordNet is missing. Enable DOWNLOAD_WORDNET to fetch the public corpus.")
    if not nltk.download("wordnet", download_dir=str(CORPUS_ROOT), quiet=True):
        raise RuntimeError("WordNet download failed.")
    wordnet.ensure_loaded()
reports = prepare_baseline_data(config, BASE_PATH)
for report in reports:
    print(report["condition"], report["setting"], report["seed"], report["rows"], "rows")

## Package the release for the team

The ZIP contains the official data, augmentation datasets, and provenance reports.
Copy it to the agreed shared folder, extract there, and verify its inventory.
Model weights are not part of this data-only package.

In [ ]:
subprocess.run([sys.executable, str(PROJECT_ROOT / "scripts/package_data_release.py"),
                "--config", str(CONFIG_PATH), "--base-path", str(BASE_PATH)], check=True)